# Profiling in PyTorch (Part 2) — `nn.Linear` to a fused MLP

A demo of **roclens Cell Profile**, based on the Hugging Face post [Profiling in PyTorch (Part 2)](https://huggingface.co/blog/torch-mlp-fusion).

Same workflow as Part 1: run with `%%rocprofv3` (or the **Cell Profile** button), then read the **Findings / Timeline / Dispatch chain / Decoded kernels** in the ROCm sidebar.

> Run the cells top to bottom — cell 3 reuses the model from cell 2.

## 0. Setup & GPU check

In [ ]:
%load_ext roclens

In [ ]:
import torch

print(f"torch.cuda.is_available() = {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device = {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No ROCm GPU visible to PyTorch — fix the runtime before profiling.")

## 1. A single `nn.Linear` — bias folded into the GEMM

`nn.Linear` dispatches **`aten::addmm`**, so there is *no separate* `aten::add`: the bias is a GEMM epilogue. Check the **Dispatch chain** (`aten::linear → aten::addmm`) and the **Decoded** kernel column (dtype / tile / layout).

In [ ]:
%%rocprofv3 --label "nn.Linear (eager)" --shapes
import torch
import torch.nn as nn

batch, in_dim, out_dim = 1024, 1024, 4096
layer = nn.Linear(in_dim, out_dim, bias=True).cuda().to(torch.bfloat16)
x = torch.randn(batch, in_dim, device="cuda", dtype=torch.bfloat16)

for _ in range(10):
    y = layer(x)
torch.cuda.synchronize()
print("linear done", y.shape)

## 2. A GeGLU MLP (eager) — 3 GEMMs + GeLU + mul

Three `nn.Linear`s with a GeGLU activation. Expect ~5 GPU kernels: two GEMMs (gate/up share a tile), a third GEMM with a different tile (down), plus pointwise **GeLU** and **mul**. Look at the **GPU kernels** table and the **Timeline**.

In [ ]:
%%rocprofv3 --label "GeGLU MLP (eager)" --shapes
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleGeGLUMLP(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden, bias=False)
        self.up_proj = nn.Linear(dim, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        g = self.gate_proj(x)
        u = self.up_proj(x)
        h = F.gelu(g, approximate="tanh")
        return self.down_proj(h * u)

batch, seq, dim, hidden = 64, 128, 768, 3072
mlp = SimpleGeGLUMLP(dim, hidden).cuda().to(torch.bfloat16)
x = torch.randn(batch, seq, dim, device="cuda", dtype=torch.bfloat16)

for _ in range(5):
    y = mlp(x)
torch.cuda.synchronize()
print("eager MLP done", y.shape)

## 3. The same MLP with `torch.compile`

Inductor fuses **GeLU + mul (+ reshape)** into **one** Triton pointwise kernel, so the big intermediate never round-trips through HBM. The 3 GEMMs stay the same. Compare **GPU kernels** / **Timeline** with cell 2: look for a `triton_poi_fused_*` kernel and fewer pointwise launches.

In [ ]:
%%rocprofv3 --label "GeGLU MLP (compiled)" --trace
import torch

# Reuses `mlp` and `x` from cell 2 (run that first).
mlp_c = torch.compile(mlp)
for _ in range(5):  # first iter compiles (cold start)
    y = mlp_c(x)
torch.cuda.synchronize()
print("compiled MLP done", y.shape)

## What to look for

- **Dispatch chain** (cell 1): `aten::linear → aten::addmm`, no separate `add`.
- **Findings** should report *compute-bound* for the MLP cells.
- Cell 2 vs 3: the two eager pointwise kernels (GeLU, mul) collapse into one fused Triton kernel under compile.
- Switch between **Tables / Timeline / Dispatch chain** via the inline **Timeline / diagnostics** link or **ROCm GPU Monitor → Cell Profile**.